VAE viewer notebook for JetBot
===

This notebook can visualize reconstructioned image by vae. This repository using JetBot camera.

In [1]:
import sys
import PIL
import numpy as np
import cv2
import traitlets
import ipywidgets.widgets as widgets
from IPython.display import display
import torch
from torchvision.transforms import transforms
from jetbot import Camera, bgr8_to_jpeg
from learning_racer.vae import VAE

## Setting Parameter

|Name | Description| Default|
|:----|:-----------|:-------|
|IMAGE_CHANNELS | Image channel such as RGB | 3 Not change|
|VARIANTS_SIZE  | Variants size of VAE      | 32          |
|MODEL_PATH     | Trained VAE model file path | ../../vae.torch|

In [2]:
IMAGE_CHANNELS = 3
VARIANTS_SIZE = 128
MODEL_PATH = '../../../vae_improved1.torch'

## Load trained VAE model.
Loading trained VAE model on GPU memory. 

In [3]:
device = torch.device('cuda')
vae = VAE(image_channels=IMAGE_CHANNELS, z_dim=VARIANTS_SIZE)
vae.load_state_dict(torch.load(MODEL_PATH, map_location=torch.device(device)), strict=False)
vae.to(device).eval()

VAE(
  (encoder): Sequential(
    (0): Conv2d(3, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (4): ReLU()
    (5): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (6): ReLU()
    (7): Conv2d(128, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (8): ReLU()
  )
  (fc_mu): Linear(in_features=12800, out_features=128, bias=True)
  (fc_logvar): Linear(in_features=12800, out_features=128, bias=True)
  (decoder_input): Linear(in_features=128, out_features=12800, bias=True)
  (decoder): Sequential(
    (0): ConvTranspose2d(256, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (1): ReLU()
    (2): ConvTranspose2d(128, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (3): ReLU()
    (4): Dropout(p=0.1, inplace=False)
    (5): ConvTranspose2d(64, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))


## Create camera 


In [4]:
camera = Camera.instance(width=320, height=240)

## Define preprocess and postprocess

In [5]:
def preprocess(image):
    observe = PIL.Image.fromarray(image)
    observe = observe.resize((160,120))
    croped = observe.crop((0, 40, 160, 120))
    tensor = transforms.ToTensor()(croped)
    return tensor
    

def rgb8_to_jpeg(image):
    return bytes(cv2.imencode('.jpg', image)[1])

## Visualize latent space function

In [6]:
ABS_LATENT_MAX_VALUE = 3
PANEL_HEIGHT = 10
PANEL_WIDTH = 10

def sigmoid(x, gain=1, offset_x=0):
    return ((np.tanh(((x+offset_x)*gain)/2)+1)/2)

def color_bar_rgb(x):
    gain = 10
    offset_x= 0.2
    offset_green = 0.6
    x = (x * 2) - 1
    red = sigmoid(x, gain, -1*offset_x)
    blue = 1-sigmoid(x, gain, offset_x)
    green = sigmoid(x, gain, offset_green) + (1-sigmoid(x,gain,-1*offset_green))
    green = green - 1.0
    return [blue * 255,green * 255,red * 255]

def _get_color(value):
    t = (value + ABS_LATENT_MAX_VALUE) / (ABS_LATENT_MAX_VALUE * 2.0)
    color = color_bar_rgb(t)
    return color

def create_color_panel(latent_spaces):
    images = []
    for z in latent_spaces:
        p = np.zeros((PANEL_HEIGHT, PANEL_WIDTH, 3))
        color = _get_color(z)
        p += color[::-1]
        p = np.clip(p, 0, 255)
        images.append(p)
    panel = np.concatenate(images, axis=1)
    return panel

#Create GUI

In [7]:
image = widgets.Image(format='jpeg', width=320, height=240)
resize = widgets.Image(format='jpeg', width=160, height=80)
result = widgets.Image(format='jpeg', width=160, height=80)
camera_link = traitlets.dlink((camera,'value'), (image,'value'), transform=bgr8_to_jpeg)
color_bar = widgets.Image(format='jpeg', width=VARIANTS_SIZE*PANEL_WIDTH, height=10*PANEL_HEIGHT)
bce_result = widgets.FloatText()
display(image)
display(widgets.HBox([resize,result]))
display(color_bar)
display(bce_result)

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

Image(value=b'', format='jpeg', height='100', width='1280')

FloatText(value=0.0)

## Start main process

In [8]:
import torch.nn.functional as F
def vae_process(change):
    image = change['new']
    image = preprocess(image)
    resize.value = rgb8_to_jpeg(np.transpose(np.uint8(image*255),[1,2,0]))
    image = torch.unsqueeze(image, dim=0).to(device)
    z, _ ,_ = vae.encode(image)
    reconst = vae.decode(z)
    to_visualize = torch.squeeze(reconst).detach().cpu().numpy()
    to_visualize = np.transpose(np.uint8(to_visualize*255),[1,2,0])[:,:,::-1]
    result.value = rgb8_to_jpeg(to_visualize)
    latent_space = z.detach().cpu().numpy()[0]
    color_bar.value = rgb8_to_jpeg(create_color_panel(latent_space))

    # Simplified loss without sigma (use BCE or MSE as placeholder; adjust if needed for visualization)
    m_vae_loss = F.binary_cross_entropy(reconst, image, reduction='sum')  # Or use ((reconst - image) ** 2).mean()
    bce_result.value = m_vae_loss.item()

vae_process({'new': camera.value})
camera.observe(vae_process, names='value')

## Cleanup process

In [9]:
camera.unobserve(vae_process, names='value')
camera.stop()
camera_link.unlink()